# 01 EDA: CCPD2019 sample
Runs on Colab (Drive mounted). Data-prep logic lives in `src/alpr/ccpd.py`; plots and tables live here.
CCPD license: Zenodo says CC BY 4.0, original repo terms unchecked, so this notebook shows statistics and plots only, no sample images.

In [1]:
# Parameters
SEED = 0
DRIVE = "/content/drive/MyDrive/alpr"
ARCHIVES = f"{DRIVE}/archives"
ARCHIVE = "/content/CCPD2019.tar.xz"        # local VM disk (fast)
SAMPLE_TAR = "/content/ccpd_sample.tar"     # built locally, copied to Drive at the end
MANIFEST = f"{ARCHIVES}/ccpd_manifest.csv"  # small file, stays on Drive
TRAIN_PER = 2200  # per subset (8 subsets); changing quotas needs no new archive pass: delete MANIFEST and rerun 4b
HELDOUT = 300      # total, spread evenly over subsets
LISTING_LOCAL = "/content/ccpd_listing.txt"          # full member listing, read by 4b when present
LISTING_DRIVE = f"{ARCHIVES}/ccpd_listing.txt"       # copy that survives a recycled VM

## Setup

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%%bash
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition

Already up to date.


In [4]:
import os, shutil
from alpr import ccpd  # ImportError on the first run? Restart the kernel once: an editable install is not visible to an already-running kernel.

os.makedirs(ARCHIVES, exist_ok=True)
print("alpr.ccpd:", ccpd.__file__)
print("local disk free GB:", round(shutil.disk_usage("/content").free / 1e9, 1))
print("Drive free GB:", round(shutil.disk_usage("/content/drive").free / 1e9, 1))
print("archive on Drive:", os.path.exists(ARCHIVE))

alpr.ccpd: /content/vehicle-plate-recognition/src/alpr/ccpd.py
local disk free GB: 101.6
Drive free GB: 96.5
archive on Drive: True


## Step 4: data prep
Each cell is safe to rerun: download resumes from the partial file (lost if the VM recycles) and verifies MD5, sample and build skip if their output exists (build writes to `.part` first). Keep the Colab session connected while they run.

In [5]:
# 4a. Download CCPD2019 (13.2 GB) to the VM's local disk and verify MD5
!python -u -m alpr.ccpd download "{ARCHIVE}"

/content/CCPD2019.tar.xz exists (13.2 GB), checking MD5...
/content/CCPD2019.tar.xz: MD5 ok (0dfbca0e6fcb7cb8ea720b0eae94c735)


In [5]:
# 4b. Pass 1: list the archive once (cached), check the subfolders, draw the stratified sample
!python -u -m alpr.ccpd sample "{ARCHIVE}" "{MANIFEST}" --seed {SEED} --train-per {TRAIN_PER} --heldout {HELDOUT} --cache "{LISTING_LOCAL}" "{LISTING_DRIVE}"

listing read from /content/ccpd_listing.txt: 355013 names
subfolders: {'ccpd_fn': 20967, 'ccpd_challenge': 50003, 'ccpd_weather': 9999, 'ccpd_blur': 20611, 'ccpd_np': 3036, 'ccpd_db': 10132, 'ccpd_rotate': 10053, 'ccpd_base': 199996, 'ccpd_tilt': 30216}
dropped unparseable: 0; duplicate files collapsed: 41495
per subset/split: {('ccpd_base', 'heldout'): 38, ('ccpd_base', 'train'): 2200, ('ccpd_blur', 'heldout'): 38, ('ccpd_blur', 'train'): 2200, ('ccpd_challenge', 'heldout'): 38, ('ccpd_challenge', 'train'): 2200, ('ccpd_db', 'heldout'): 38, ('ccpd_db', 'train'): 2200, ('ccpd_fn', 'heldout'): 37, ('ccpd_fn', 'train'): 2200, ('ccpd_rotate', 'heldout'): 37, ('ccpd_rotate', 'train'): 2200, ('ccpd_tilt', 'heldout'): 37, ('ccpd_tilt', 'train'): 2200, ('ccpd_weather', 'heldout'): 37, ('ccpd_weather', 'train'): 2200}


In [6]:
# 4c. Pass 2: stream the archive again, copy only the sampled images into one local .tar (+ manifest.csv); the last cell copies the sample to Drive (the archive copy is optional)
!python -u -m alpr.ccpd build "{ARCHIVE}" "{MANIFEST}" "{SAMPLE_TAR}"

  2000/17900 copied
  4000/17900 copied
  6000/17900 copied
  8000/17900 copied
  10000/17900 copied
  12000/17900 copied
  14000/17900 copied
  16000/17900 copied
/content/ccpd_sample.tar: MD5 5019a5333b4a92aded34cd3fb52eff4c


In [7]:
# 4d. Verify the packed sample against the manifest
import collections, tarfile
rows = ccpd.read_manifest(MANIFEST)
with tarfile.open(SAMPLE_TAR) as tf:
    names = [n for n in tf.getnames() if n != "manifest.csv"]
print("manifest rows:", len(rows), "| images in tar:", len(names))
print("tar per split:", dict(collections.Counter(n.split("/")[0] for n in names)))
assert sorted(n.split("/", 1)[1] for n in names) == sorted(m.rsplit("/", 1)[1] for m, _, _ in rows)
print("sample tar:", round(os.path.getsize(SAMPLE_TAR) / 1e9, 2), "GB, MD5", ccpd.file_md5(SAMPLE_TAR))

manifest rows: 17900 | images in tar: 17900
tar per split: {'heldout': 300, 'train': 17600}
sample tar: 1.26 GB, MD5 5019a5333b4a92aded34cd3fb52eff4c


In [8]:
!cp /content/ccpd_sample.tar {ARCHIVES}/ccpd_sample.tar
!md5sum /content/ccpd_sample.tar {ARCHIVES}/ccpd_sample.tar

5019a5333b4a92aded34cd3fb52eff4c  /content/ccpd_sample.tar
5019a5333b4a92aded34cd3fb52eff4c  /content/drive/MyDrive/alpr/archives/ccpd_sample.tar


## EDA
Tables only, no sample images (CCPD license handling).

In [9]:
# 5a. Duplicates across subfolders, from the cached listing (no archive pass)
import pandas as pd
from pathlib import PurePosixPath

names = open(next(p for p in (LISTING_LOCAL, LISTING_DRIVE) if os.path.exists(p))).read().splitlines()
df = pd.DataFrame({"member": names})
df["subset"] = df.member.map(lambda n: PurePosixPath(n).parent.name)
df["base"] = df.member.map(lambda n: PurePosixPath(n).name)
per_name = df.groupby("base").agg(n_files=("member", "size"), n_subsets=("subset", "nunique"))
df["dup"] = df.base.map(per_name.n_files) > 1
print(f"{len(df):,} files, {len(per_name):,} unique names; names with >1 file: {(per_name.n_files > 1).sum():,} "
      f"({df.dup.sum():,} files, {df.dup.mean():.1%})")
print("\nShare of each subfolder's files in a duplicated name:")
print(df.groupby("subset").dup.mean().sort_values(ascending=False).map("{:.0%}".format).to_string())
print("\nNames by number of files (rows) and number of subfolders (columns):")
print(pd.crosstab(per_name.n_files, per_name.n_subsets))
present = pd.crosstab(df.base, df.subset).clip(upper=1)
print("\nNames shared by each pair of subfolders (diagonal = unique names in the subfolder):")
print((present.T @ present).to_string())

kept, removed = ccpd.dedup([n for n in names if PurePosixPath(n).parent.name != ccpd.EXCLUDED])
avail = pd.Series([PurePosixPath(n).parent.name for n in kept]).value_counts()
need = TRAIN_PER + HELDOUT // len(ccpd.SUBSETS) + 1
print(f"\nAfter dedup (np excluded): {len(kept):,} unique images, {removed:,} files collapsed.")
print(f"Available per subset label (need {need}):")
print(avail.reindex(ccpd.PRIORITY).to_string())

355,013 files, 313,518 unique names; names with >1 file: 34,917 (76,412 files, 21.5%)

Share of each subfolder's files in a duplicated name:
subset
ccpd_blur         72%
ccpd_db           67%
ccpd_fn           67%
ccpd_rotate       61%
ccpd_challenge    48%
ccpd_tilt         35%
ccpd_weather       1%
ccpd_base          0%
ccpd_np            0%

Names by number of files (rows) and number of subfolders (columns):
n_subsets       1      2     3    4   5   6
n_files                                    
1          278601      0     0    0   0   0
2               0  28989     0    0   0   0
3               0      0  5384    0   0   0
4               0      0     0  454   0   0
5               0      0     0    0  74   0
6               0      0     0    0   0  16

Names shared by each pair of subfolders (diagonal = unique names in the subfolder):
subset          ccpd_base  ccpd_blur  ccpd_challenge  ccpd_db  ccpd_fn  ccpd_np  ccpd_rotate  ccpd_tilt  ccpd_weather
subset                        